In [0]:
%pip install -U -qq langchain-databricks langchain==0.3.7 langchain_community==0.3.5 flashrank
dbutils.library.restartPython()

In [0]:
from langchain_databricks import ChatDatabricks
from langchain.agents import initialize_agent, Tool
import pandas as pd
import json
from databricks.vector_search.client import VectorSearchClient
import mlflow.deployments

# Define the LLM
llm_llama = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=2500)

# Dataset knowledge
dataset_knowledge = """
This dataset contains campaign performance metrics for various brands.
Each row includes:
- `kpi_metric_name`: the type of KPI being measured (e.g., Brand Familiarity, Ad Recall)
- `mapping`: indicates the channel grouping (e.g., Digital, LTV, Total Campaign)
- `control_pct`: percentage of behavior observed in the control group
- `exposed_pct`: percentage in the exposed group
- `lift_pct`: difference between exposed and control indicating the lift
- `metric_display_name`: readable name for the KPI
"""

# Vector search configuration
catalog = "solutions_catalog"
schema = "pih_schema"
index_name = "row_text_self_managed_vs_index"
vs_endpoint_name = "vs_endpoint_pih_brand"
index_full = f"{catalog}.{schema}.{index_name}"
vsc = VectorSearchClient(disable_notice=True)

# Embedding client using MLflow deployments
deploy_client = mlflow.deployments.get_deploy_client("databricks")

def query_study_id_llm_based(study_id, user_query, top_k=10):
    try:
        # Create embedding for the user query using MLflow deployment
        response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [user_query]})
        embeddings = [e["embedding"] for e in response.data]

        # Perform vector search using the embedding
        vs_index = vsc.get_index(endpoint_name=vs_endpoint_name, index_name=index_full)
        results = vs_index.similarity_search(
            query_vector=embeddings[0],
            columns=["study_id", "kpi_metric_name", "lift_pct", "metric_display_name", "mapping"],
            num_results=top_k
        )

        passages = []
        for doc in results.get("result", {}).get("data_array", []):
            new_doc = {
                "study_id": doc[0],
                "kpi_metric_name": doc[1],
                "lift_pct": doc[2],
                "metric_display_name": doc[3],
                "mapping": doc[4]
            }
            passages.append(new_doc)

            print(passages)

        df = pd.DataFrame(passages)
        return df

    except Exception as e:
        print(f"LLM similarity search error: {e}")
        return pd.DataFrame()

def summarize_rag(user_input: str) -> str:
    try:
        study_id = ''.join(filter(str.isdigit, user_input))
        df = query_study_id_llm_based(study_id, user_input)
        if df.empty:
            return f"No relevant rows found for Study ID {study_id}"

        data_preview = df.head(10).to_dict(orient="records")
        prompt = f"""
You are a campaign analyst assistant. Here is the knowledge of the dataset:
{dataset_knowledge}

Instructions:
Summarize the KPI metrics with the highest and lowest average lift_pct. Highlight any interesting patterns based on mapping and control/exposed pct.

Use the following relevant data to reason and summarize:
{json.dumps(data_preview, indent=2)}

Give a concise and insightful summary with bullet points if needed.
"""
        response = llm_llama.invoke(prompt)
        return response.content.strip()
    except Exception as e:
        return f"Error summarizing for Study ID {study_id}: {str(e)}"
    


def detect_anomalies_rag(user_input: str) -> str:
    try:
        study_id = ''.join(filter(str.isdigit, user_input))
        df = query_study_id_llm_based(study_id, user_input)
        if df.empty:
            return f"No relevant rows found for Study ID {study_id}"

        df["z_score"] = (df["lift_pct"] - df["lift_pct"].mean()) / df["lift_pct"].std()
        anomalies = df[abs(df["z_score"]) > 2]
        if anomalies.empty:
            return "No significant anomalies detected in lift_pct."
        else:
            return f"Detected {len(anomalies)} anomalies in lift_pct:\n" + anomalies[["kpi_metric_name", "mapping", "lift_pct", "z_score"]].to_string(index=False)
    except Exception as e:
        return f"Error detecting anomalies: {str(e)}"
    

def brand_lift_by_kpi(user_input: str) -> str:
    try:
        # Use embedding for the KPI metric input
        response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [user_input]})
        embeddings = [e["embedding"] for e in response.data]

        # Vector search on the full index
        vs_index = vsc.get_index(endpoint_name=vs_endpoint_name, index_name=index_full)
        results = vs_index.similarity_search(
            query_vector=embeddings[0],
            columns=["study_id", "kpi_metric_name", "lift_pct", "metric_display_name", "mapping"],
            num_results=10
        )

        # Format results into DataFrame
        passages = []
        for doc in results.get("result", {}).get("data_array", []):
            passages.append({
                "study_id": doc[0],
                "kpi_metric_name": doc[1],
                "lift_pct": doc[2],
                "metric_display_name": doc[3],
                "mapping": doc[4]
            })

        if not passages:
            return f"No relevant data found for KPI metric: {user_input}"

        df = pd.DataFrame(passages)

        # Build prompt and summarize
        prompt = f"""
            You are a brand lift analyst. The dataset below contains campaign performance related to a KPI metric similar to: "{user_input}".

            Each row includes:
            - KPI name
            - Lift percentage (difference between exposed and control)
            - Mapping (channel category like Digital, LTV, etc.)

            Analyze and summarize the brand lift performance across mappings. Mention which mappings have high/low lift for this KPI, and any visible trends.

            Relevant data:
            {json.dumps(df.to_dict(orient="records"), indent=2)}

            Give a concise summary with insights.
            """

        response = llm_llama.invoke(prompt)
        return response.content.strip()

    except Exception as e:
        return f"Error analyzing brand lift for KPI metric: {str(e)}"
    

def compare_platforms_performance(user_input: str) -> str:
    try:
        # Parse platform names from input
        if " vs " not in user_input.lower():
            return "Please format your query like: 'Compare Digital vs LTV Only'."

        platform_1, platform_2 = map(str.strip, user_input.split("vs"))

        # Generate embeddings for each platform name
        response_1 = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [platform_1]})
        response_2 = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [platform_2]})
        emb_1 = response_1.data[0]["embedding"]
        emb_2 = response_2.data[0]["embedding"]

        # Vector search on platform-related results
        vs_index = vsc.get_index(endpoint_name=vs_endpoint_name, index_name=index_full)

        results_1 = vs_index.similarity_search(
            query_vector=emb_1,
            columns=["study_id", "platform", "lift_pct", "metric_display_name", "mapping", "kpi_metric_name"],
            num_results=10
        )
        results_2 = vs_index.similarity_search(
            query_vector=emb_2,
            columns=["study_id", "platform", "lift_pct", "metric_display_name", "mapping", "kpi_metric_name"],
            num_results=10
        )

        def extract_rows(results):
            return [{
                "study_id": r[0],
                "platform": r[1],
                "lift_pct": r[2],
                "metric_display_name": r[3],
                "mapping": r[4],
                "kpi_metric_name": r[5]
            } for r in results.get("result", {}).get("data_array", [])]

        data_1 = extract_rows(results_1)
        data_2 = extract_rows(results_2)

        if not data_1 or not data_2:
            return "Could not find enough data for one or both platforms."

        # Create comparison prompt for LLM
        prompt = f"""
            You are a campaign analyst. Compare the performance across two platforms:
            - Platform 1: {platform_1}
            - Platform 2: {platform_2}

            Each dataset contains:
            - KPI Name
            - Platform
            - Lift Percentage (lift_pct)
            - Mapping (channel type)
            - Metric Display Name

            Analyze and compare:
            - Which platform has higher average lift_pct?
            - What patterns or KPIs perform better on one platform than the other?

            Data for {platform_1}:
            {json.dumps(data_1, indent=2)}

            Data for {platform_2}:
            {json.dumps(data_2, indent=2)}

            Provide a clear summary with insights and recommendations.
            """
        response = llm_llama.invoke(prompt)
        return response.content.strip()

    except Exception as e:
        return f"Error comparing platform performance: {str(e)}"

    

def compare_kpi_performance(user_input: str) -> str:
    try:
        # Parse KPI names from the input using 'vs' as the separator
        if " vs " not in user_input.lower():
            return "Please format your query like: 'Compare Brand Awareness vs Ad Recall'."

        kpi_1, kpi_2 = map(str.strip, user_input.split("vs"))

        # Generate embeddings for each KPI
        response_1 = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [kpi_1]})
        response_2 = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [kpi_2]})
        emb_1 = response_1.data[0]["embedding"]
        emb_2 = response_2.data[0]["embedding"]

        # Vector search for both KPI names
        vs_index = vsc.get_index(endpoint_name=vs_endpoint_name, index_name=index_full)

        results_1 = vs_index.similarity_search(
            query_vector=emb_1,
            columns=["study_id", "kpi_metric_name", "lift_pct", "metric_display_name", "mapping"],
            num_results=10
        )
        results_2 = vs_index.similarity_search(
            query_vector=emb_2,
            columns=["study_id", "kpi_metric_name", "lift_pct", "metric_display_name", "mapping"],
            num_results=10
        )

        def extract_passages(results):
            return [{
                "study_id": r[0],
                "kpi_metric_name": r[1],
                "lift_pct": r[2],
                "metric_display_name": r[3],
                "mapping": r[4]
            } for r in results.get("result", {}).get("data_array", [])]

        data_1 = extract_passages(results_1)
        data_2 = extract_passages(results_2)

        if not data_1 or not data_2:
            return "Could not find enough relevant data for one or both KPI metrics."

        # Prepare prompt
        prompt = f"""
            You are a campaign performance analyst. Your task is to compare and contrast the performance of these two KPIs:
            - KPI 1: {kpi_1}
            - KPI 2: {kpi_2}

            Each dataset below contains:
            - KPI name
            - Lift percentage (lift_pct)
            - Mapping (channel category)
            - Study ID

            Analyze the difference in average lift, best-performing channels, and any trend differences between the two KPIs.

            Data for {kpi_1}:
            {json.dumps(data_1, indent=2)}

            Data for {kpi_2}:
            {json.dumps(data_2, indent=2)}

            Return a concise but insightful summary with bullet points.
            """
        response = llm_llama.invoke(prompt)
        return response.content.strip()

    except Exception as e:
        return f"Error comparing KPI performance: {str(e)}"



def campaign_optimization_rag(user_input: str) -> str:
    try:
        study_id = ''.join(filter(str.isdigit, user_input))
        df = query_study_id_llm_based(study_id, user_input)
        if df.empty:
            return f"No relevant rows found for Study ID {study_id}"

        avg_lift_by_mapping = df.groupby("mapping")["lift_pct"].mean().reset_index()
        avg_lift_by_mapping = avg_lift_by_mapping.sort_values(by="lift_pct", ascending=False)
        top_channel = avg_lift_by_mapping.iloc[0]
        bottom_channel = avg_lift_by_mapping.iloc[-1]
        return "\n".join([
            f"**Study ID: {study_id}**",
            "**Channel Performance Summary**:",
            avg_lift_by_mapping.to_string(index=False),
            "",
            f"**Recommended Channel to Invest More In**: **{top_channel['mapping']}** (Avg Lift: {top_channel['lift_pct']:.2f}%)",
            f"**Channel to Monitor/Possibly Reconsider**: **{bottom_channel['mapping']}** (Avg Lift: {bottom_channel['lift_pct']:.2f}%)"
        ])
    except Exception as e:
        return f"Error in campaign optimization: {str(e)}"

# Register tools
tools = [
    Tool(name="summarize_rag", func=summarize_rag, description="Summarize metrics for a given Study ID using RAG."),
    Tool(name="detect_anomalies_rag", func=detect_anomalies_rag, description="Detect anomalies in lift_pct for a given Study ID using RAG."),
    Tool(name="campaign_optimization_rag", func=campaign_optimization_rag, description="Optimize channel performance using RAG for a given Study ID."),
    Tool(name="brand_lift_by_kpi",    func=brand_lift_by_kpi,    description="Analyze and summarize brand lift trends based on KPI metric name input (e.g., Ad Recall, Brand Awareness)."),
    Tool(name="compare_kpi_performance",func=compare_kpi_performance,description="Compare performance between two KPI metrics using brand lift and channel mapping (format: 'Compare KPI1 vs KPI2')."),
    Tool(    name="compare_platforms_performance",    func=compare_platforms_performance,    description="Compare campaign performance across two platforms like 'Digital vs LTV Only'.")
]

# Initialize agent
agent = initialize_agent(tools=tools, llm=llm_llama, agent="zero-shot-react-description", verbose=True)




In [0]:
# Main run
if __name__ == "__main__":
    print("\n--- Run: Summarization via LLM Similarity Search ---")
    ##print(agent.run("Summarize the KPI data for KPI is Allstate is a good fit for ABC"))

    print(agent.run("Summarize the KPI data for KPI is Offers quality products/services"))

In [0]:
# Main run
if __name__ == "__main__":
    print("\n--- Run: Summarization via LLM Similarity Search ---")
    ##print(agent.run("Summarize the KPI data for KPI is Allstate is a good fit for ABC"))

    print(agent.run("Summarize the KPI data for KPI is Unaided Brand Awareness"))
    
 

In [0]:
# Main run
if __name__ == "__main__":
    print("\n--- Run: Summarization via LLM Similarity Search ---")
    ##print(agent.run("Summarize the KPI data for KPI is Allstate is a good fit for ABC"))

    print(agent.run("Which metric — like Ad Recall or Brand Familiarity — shows the largest deviation?"))
    
    


In [0]:
# Main run
if __name__ == "__main__":
    print("\n--- Run: Summarization via LLM Similarity Search ---")
    ##print(agent.run("Summarize the KPI data for KPI is Allstate is a good fit for ABC"))

    print(agent.run("Compare Brand Familiarity vs Ad Recall"))

In [0]:
# Main run
if __name__ == "__main__":
    print("\n--- Run: Summarization via LLM Similarity Search ---")
    ##print(agent.run("Summarize the KPI data for KPI is Allstate is a good fit for ABC"))

    print(agent.run("Compare Trust in brand vs Is relevant to me"))





In [0]:

# Main run
if __name__ == "__main__":
    print("\n--- Run: Summarization via LLM Similarity Search ---")
    ##print(agent.run("Summarize the KPI data for KPI is Allstate is a good fit for ABC"))

    print(agent.run("Compare Digital vs LTV Only"))
    
    